# Charge Deposition Benchmark: CIC vs SpaceChargeKick

This notebook benchmarks the new Cloud-in-Cell (CIC) charge deposition functions against the existing `_deposit_charge_on_grid` method from the `SpaceChargeKick` class. We'll compare performance, accuracy, memory usage, and scaling behavior.

## Overview

- **New Implementation**: `deposit_charge_cic` functions from `cheetah.utils.charge_deposition`
- **Current Implementation**: `SpaceChargeKick._deposit_charge_on_grid`
- **Comparison Metrics**: Execution time, memory usage, scaling behavior, numerical accuracy

## 1. Import Required Libraries

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
import psutil
import gc
from typing import Tuple, List
from dataclasses import dataclass

# Cheetah imports
import sys
sys.path.append('../')

from cheetah.utils.charge_deposition import (
    deposit_charge_cic,
    deposit_charge_cic_1d, 
    deposit_charge_cic_2d,
    deposit_charge_cic_3d
)
from cheetah.accelerator.space_charge_kick import SpaceChargeKick
from cheetah.particles import ParticleBeam, Species

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 2. Setup Test Data Generation

In [ ]:
@dataclass 
class BenchmarkResult:
    """Store comprehensive benchmark timing results with statistics"""
    method_name: str
    num_particles: int
    grid_shape: Tuple[int, ...]
    batch_size: int
    execution_time_mean: float
    execution_time_std: float
    execution_time_min: float
    execution_time_max: float
    memory_usage_mean: float
    memory_usage_std: float
    memory_usage_min: float
    memory_usage_max: float
    num_trials: int

@dataclass
class TimingStats:
    """Statistical analysis of timing measurements"""
    mean: float
    std: float
    min: float
    max: float
    median: float
    q25: float
    q75: float
    sem: float  # Standard error of the mean
    ci_lower: float  # 95% confidence interval lower bound
    ci_upper: float  # 95% confidence interval upper bound

def compute_timing_statistics(times: List[float]) -> TimingStats:
    """Compute comprehensive statistics for timing measurements"""
    import numpy as np
    from scipy import stats
    
    times_array = np.array(times)
    n = len(times_array)
    
    mean_val = np.mean(times_array)
    std_val = np.std(times_array, ddof=1) if n > 1 else 0.0
    sem_val = std_val / np.sqrt(n) if n > 1 else 0.0
    
    # 95% confidence interval using t-distribution
    if n > 1:
        t_critical = stats.t.ppf(0.975, df=n-1)
        margin_error = t_critical * sem_val
        ci_lower = mean_val - margin_error
        ci_upper = mean_val + margin_error
    else:
        ci_lower = ci_upper = mean_val
    
    return TimingStats(
        mean=mean_val,
        std=std_val,
        min=np.min(times_array),
        max=np.max(times_array),
        median=np.median(times_array),
        q25=np.percentile(times_array, 25),
        q75=np.percentile(times_array, 75),
        sem=sem_val,
        ci_lower=ci_lower,
        ci_upper=ci_upper
    )

class TestDataGenerator:
    """Generate test data for benchmarking charge deposition methods"""
    
    @staticmethod
    def create_test_beam(
        num_particles: int,
        batch_size: int = 1,
        energy: float = 1e6,  # eV
        charge: float = 1e-12,  # Coulombs
        device: torch.device = device,
        offset: torch.tensor = None
    ) -> ParticleBeam:
        """Create a test particle beam with Gaussian distributions"""
        
        # Create particle coordinates (x, px, y, py, tau, delta, s)
        particles = torch.zeros(batch_size, num_particles, 7, device=device)
        
        if num_particles > 1:
            # Use Gaussian distributions for realistic beam profiles
            sigma_x = 1e-3  # 1 mm
            sigma_y = 1e-3  # 1 mm  
            sigma_tau = 1e-3  # 1 mm
            sigma_px = 1e-5  # Small momentum spread
            sigma_py = 1e-5
            sigma_delta = 1e-3  # Small energy spread
            
            # Generate Gaussian distributions
            particles[..., 0] = torch.randn(batch_size, num_particles, device=device) * sigma_x  # x
            particles[..., 1] = torch.randn(batch_size, num_particles, device=device) * sigma_px # px
            particles[..., 2] = torch.randn(batch_size, num_particles, device=device) * sigma_y  # y
            particles[..., 3] = torch.randn(batch_size, num_particles, device=device) * sigma_py # py
            particles[..., 4] = torch.randn(batch_size, num_particles, device=device) * sigma_tau # tau
            particles[..., 5] = torch.randn(batch_size, num_particles, device=device) * sigma_delta # delta
            particles[..., 6] = torch.zeros(batch_size, num_particles, device=device)  # s
        
        offset = offset or torch.zeros(7)
        offset = offset.to(device)
        particles[...,:] += offset

        # Create uniform particle charges and survival probabilities
        particle_charges = torch.full(
            (batch_size, num_particles), charge, device=device
        )
        survival_probabilities = torch.ones(
            (batch_size, num_particles), device=device
        )
        
        beam = ParticleBeam(
            particles=particles,
            energy=torch.full((batch_size,), energy, device=device),
            particle_charges=particle_charges,
            survival_probabilities=survival_probabilities,
            device=device
        )
        
        return beam
    
    @staticmethod
    def create_grid_parameters(
        grid_shape: Tuple[int, int, int],
        beam: ParticleBeam,
        grid_extent: float = 5.0
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Create grid dimensions and cell sizes based on beam parameters"""
        
        if beam.particles.shape[-2] == 1:
            # Special case: single particle, set small grid dimensions
            grid_dimensions = torch.tensor([1e-2, 1e-2, 1e-2], device=beam.particles.device, dtype=beam.particles.dtype)
            cell_size = 2 * grid_dimensions / torch.tensor(grid_shape, device=beam.particles.device, dtype=beam.particles.dtype)
            return grid_dimensions.unsqueeze(0), cell_size

        else:
            # Calculate beam dimensions (using sigma values)
            grid_dimensions = torch.stack([
                grid_extent * beam.sigma_x,
                grid_extent * beam.sigma_y, 
                grid_extent * beam.sigma_tau
            ], dim=-1)
            
            cell_size = (
                2 * grid_dimensions / 
                torch.tensor(grid_shape, device=beam.particles.device, dtype=beam.particles.dtype)
            )
            
            return grid_dimensions, cell_size

# Test the data generation
test_beam = TestDataGenerator.create_test_beam(1000, batch_size=2)
print(f"Test beam shape: {test_beam.particles.shape}")
print(f"Beam energy: {test_beam.energy}")
print(f"Sigma x: {test_beam.sigma_x}")
print(f"Sigma y: {test_beam.sigma_y}")
print(f"Sigma tau: {test_beam.sigma_tau}")

# Test single particle case
single_particle_beam = TestDataGenerator.create_test_beam(1, batch_size=1)
print("Single particle beam:")
print(f"Particle position: {single_particle_beam.particles[0, 0, [0, 2, 4]]}")  # x, y, tau
print(f"All coordinates: {single_particle_beam.particles[0, 0, :]}")

# Import scipy for statistical analysis
try:
    import scipy.stats
    print("✅ SciPy available for statistical analysis")
except ImportError:
    print("⚠️ SciPy not available - using basic statistics only")

## 3. Benchmark Cloud-in-Cell Charge Deposition

In [ ]:
def benchmark_method_with_stats(
    method_func,
    method_name: str,
    num_particles: int,
    grid_shape: Tuple[int, int, int],
    batch_size: int = 1,
    num_trials: int = 10,
    warmup_trials: int = 2
) -> BenchmarkResult:
    """
    Benchmark a method with comprehensive statistical analysis
    
    Parameters
    ----------
    method_func : callable
        Function that performs the computation and returns the result
    method_name : str
        Name of the method for identification
    num_particles : int
        Number of particles to simulate
    grid_shape : tuple
        Shape of the grid
    batch_size : int
        Batch size for the simulation
    num_trials : int
        Number of trials for statistical analysis
    warmup_trials : int
        Number of warmup trials to exclude from statistics
    
    Returns
    -------
    BenchmarkResult
        Comprehensive benchmark results with statistics
    """
    
    execution_times = []
    memory_usages = []
    
    # Warmup runs
    for _ in range(warmup_trials):
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        
        _ = method_func()
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
    
    # Actual benchmark runs
    for trial in range(num_trials):
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        
        start_time = time.time()
        result = method_func()
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        end_time = time.time()
        execution_time = end_time - start_time
        
        memory_usage = torch.cuda.max_memory_allocated() / 1e6 if device.type == 'cuda' else 0
        
        execution_times.append(execution_time)
        memory_usages.append(memory_usage)
    
    # Compute statistics
    exec_stats = compute_timing_statistics(execution_times)
    mem_stats = compute_timing_statistics(memory_usages)
    
    return BenchmarkResult(
        method_name=method_name,
        num_particles=num_particles,
        grid_shape=grid_shape,
        batch_size=batch_size,
        execution_time_mean=exec_stats.mean,
        execution_time_std=exec_stats.std,
        execution_time_min=exec_stats.min,
        execution_time_max=exec_stats.max,
        memory_usage_mean=mem_stats.mean,
        memory_usage_std=mem_stats.std,
        memory_usage_min=mem_stats.min,
        memory_usage_max=mem_stats.max,
        num_trials=num_trials
    )

def benchmark_cic_methods(
    num_particles: int,
    grid_shape: Tuple[int, int, int],
    batch_size: int = 1,
    num_trials: int = 10
) -> List[BenchmarkResult]:
    """Benchmark the CIC charge deposition methods with statistical analysis"""
    
    results = []
    
    # Create test data once to use for all methods
    beam = TestDataGenerator.create_test_beam(num_particles, batch_size)
    grid_dimensions, cell_size = TestDataGenerator.create_grid_parameters(grid_shape, beam)
    
    # Extract particle positions for CIC functions
    xp_coords = beam.to_xyz_pxpypz()
    x = xp_coords[..., 0]  # x positions
    y = xp_coords[..., 2]  # y positions  
    z = xp_coords[..., 4]  # z positions
    
    # Create charge weights (particle charges * survival probabilities)
    weights = beam.particle_charges * beam.survival_probabilities
    
    # Create bin arrays for CIC functions
    bins_x = torch.linspace(
        -grid_dimensions[0, 0], grid_dimensions[0, 0], 
        grid_shape[0], device=device
    )
    bins_y = torch.linspace(
        -grid_dimensions[0, 1], grid_dimensions[0, 1],
        grid_shape[1], device=device  
    )
    bins_z = torch.linspace(
        -grid_dimensions[0, 2], grid_dimensions[0, 2],
        grid_shape[2], device=device
    )
    
    # Define method functions for benchmarking
    def cic_3d_func():
        return deposit_charge_cic_3d(x, y, z, bins_x, bins_y, bins_z, weights)
    
    def cic_2d_func():
        return deposit_charge_cic_2d(x, y, bins_x, bins_y, weights)
    
    def cic_1d_func():
        return deposit_charge_cic_1d(x, bins_x, weights)
    
    # Benchmark each method
    methods = [
        (cic_3d_func, "CIC_3D", grid_shape),
        (cic_2d_func, "CIC_2D", grid_shape[:2]),
        (cic_1d_func, "CIC_1D", grid_shape[:1])
    ]
    
    for method_func, method_name, shape in methods:
        print(f"Benchmarking {method_name}...")
        result = benchmark_method_with_stats(
            method_func, method_name, num_particles, shape, batch_size, num_trials
        )
        results.append(result)
        
        # Print results with confidence intervals
        print(f"  {method_name}: {result.execution_time_mean:.4f} ± {result.execution_time_std:.4f}s "
              f"(min: {result.execution_time_min:.4f}s, max: {result.execution_time_max:.4f}s)")
        print(f"  Memory: {result.memory_usage_mean:.1f} ± {result.memory_usage_std:.1f}MB "
              f"(min: {result.memory_usage_min:.1f}MB, max: {result.memory_usage_max:.1f}MB)")
    
    return results

# Run enhanced benchmark
print("Testing CIC methods with enhanced statistics (10,000 particles on 32x32x32 grid):")
print("=" * 70)
cic_results = benchmark_cic_methods(10000, (32, 32, 32), batch_size=1, num_trials=10)

## 4. Compare with SpaceChargeKick Method

In [ ]:
def benchmark_space_charge_kick_method(
    num_particles: int,
    grid_shape: Tuple[int, int, int],
    batch_size: int = 1,
    num_trials: int = 10
) -> BenchmarkResult:
    """Benchmark the existing SpaceChargeKick._deposit_charge_on_grid method with statistics"""
    
    # Create test data
    beam = TestDataGenerator.create_test_beam(num_particles, batch_size)
    grid_dimensions, cell_size = TestDataGenerator.create_grid_parameters(grid_shape, beam)
    
    # Create SpaceChargeKick instance
    space_charge_kick = SpaceChargeKick(
        effect_length=torch.tensor(0.1, device=device),
        grid_shape=grid_shape
    )
    
    # Get xp coordinates for SpaceChargeKick method
    xp_coords = beam.to_xyz_pxpypz()
    
    # Define method function
    def sck_func():
        return space_charge_kick._deposit_charge_on_grid(
            beam, xp_coords, cell_size, grid_dimensions
        )
    
    # Use enhanced benchmarking
    result = benchmark_method_with_stats(
        sck_func, "SpaceChargeKick", num_particles, grid_shape, batch_size, num_trials
    )
    
    print(f"SpaceChargeKick: {result.execution_time_mean:.4f} ± {result.execution_time_std:.4f}s "
          f"(min: {result.execution_time_min:.4f}s, max: {result.execution_time_max:.4f}s)")
    print(f"Memory: {result.memory_usage_mean:.1f} ± {result.memory_usage_std:.1f}MB "
          f"(min: {result.memory_usage_min:.1f}MB, max: {result.memory_usage_max:.1f}MB)")
    
    return result

def print_detailed_comparison(cic_results: List[BenchmarkResult], sck_result: BenchmarkResult):
    """Print detailed performance comparison with statistical significance"""
    
    print(f"\n{'='*80}")
    print("DETAILED PERFORMANCE COMPARISON WITH STATISTICS")
    print(f"{'='*80}")
    
    # Performance comparison
    print(f"\n{'Method':<20} {'Mean Time (s)':<15} {'Std Dev (s)':<12} {'Memory (MB)':<15} {'Mem Std (MB)':<12}")
    print(f"{'-'*80}")
    
    for result in cic_results:
        print(f"{result.method_name:<20} {result.execution_time_mean:<15.6f} "
              f"{result.execution_time_std:<12.6f} {result.memory_usage_mean:<15.1f} "
              f"{result.memory_usage_std:<12.1f}")
    
    print(f"{sck_result.method_name:<20} {sck_result.execution_time_mean:<15.6f} "
          f"{sck_result.execution_time_std:<12.6f} {sck_result.memory_usage_mean:<15.1f} "
          f"{sck_result.memory_usage_std:<12.1f}")
    
    # Speedup analysis
    print(f"\n{'SPEEDUP ANALYSIS (vs SpaceChargeKick)'}")
    print(f"{'-'*50}")
    
    for result in cic_results:
        speedup_mean = sck_result.execution_time_mean / result.execution_time_mean
        # Propagate uncertainty in speedup calculation
        speedup_rel_error = np.sqrt(
            (sck_result.execution_time_std / sck_result.execution_time_mean)**2 + 
            (result.execution_time_std / result.execution_time_mean)**2
        )
        speedup_std = speedup_mean * speedup_rel_error
        
        memory_ratio = result.memory_usage_mean / sck_result.memory_usage_mean if sck_result.memory_usage_mean > 0 else np.inf
        
        print(f"{result.method_name:<12}: {speedup_mean:.2f} ± {speedup_std:.2f}x speedup, "
              f"{memory_ratio:.2f}x memory ratio")
    
    # Statistical significance test (if scipy is available)
    try:
        from scipy import stats
        print(f"\n{'STATISTICAL SIGNIFICANCE TESTS'}")
        print(f"{'-'*40}")
        print("(Two-sample t-tests comparing execution times)")
        
        for result in cic_results:
            # For demonstration, we assume normally distributed times
            # In practice, you'd need the raw timing data for proper testing
            print(f"{result.method_name} vs SpaceChargeKick: p-value analysis requires raw timing data")
            
    except ImportError:
        print(f"\n{'Note: Install scipy for statistical significance testing'}")

# Benchmark SpaceChargeKick method with enhanced statistics
print("\nTesting SpaceChargeKick method with enhanced statistics (100,000 particles on 32x32x32 grid):")
print("=" * 85)
sck_result = benchmark_space_charge_kick_method(100000, (32, 32, 32), batch_size=1, num_trials=10)

# Print detailed comparison
print_detailed_comparison(cic_results, sck_result)

## 5. Accuracy Comparison and Output Verification

## 5.1. CIC vs SpaceChargeKick Accuracy Check

In [ ]:
def compare_charge_deposition_accuracy(
    num_particles: int = 100000,
    grid_shape: Tuple[int, int, int] = (16, 16, 16),
    batch_size: int = 1,
    offset: torch.Tensor = None,
) -> dict:
    """Compare the accuracy of CIC vs SpaceChargeKick charge deposition methods"""
    
    print(f"Comparing accuracy with {num_particles} particles on {grid_shape} grid...")
    
    # Create identical test data for both methods
    beam = TestDataGenerator.create_test_beam(num_particles, batch_size, offset=offset)
    grid_dimensions, cell_size = TestDataGenerator.create_grid_parameters(grid_shape, beam)
    
    # Get coordinates
    xp_coords = beam.to_xyz_pxpypz()
    x = xp_coords[..., 0]
    y = xp_coords[..., 2]
    z = xp_coords[..., 4]
    weights = beam.particle_charges * beam.survival_probabilities
    
    # Create bins for CIC method
    bins_x = torch.linspace(-grid_dimensions[0, 0], grid_dimensions[0, 0], grid_shape[0], device=device)
    bins_y = torch.linspace(-grid_dimensions[0, 1], grid_dimensions[0, 1], grid_shape[1], device=device)
    bins_z = torch.linspace(-grid_dimensions[0, 2], grid_dimensions[0, 2], grid_shape[2], device=device)
    
    # Calculate charge using CIC method
    cic_grid = deposit_charge_cic_3d(x, y, z, bins_x, bins_y, bins_z, weights)
    
    # Calculate charge using SpaceChargeKick method
    space_charge_kick = SpaceChargeKick(
        effect_length=torch.tensor(0.1, device=device),
        grid_shape=grid_shape
    )
    sck_grid = space_charge_kick._deposit_charge_on_grid(beam, xp_coords, cell_size, grid_dimensions)
    
    # Calculate difference statistics
    abs_diff = torch.abs(cic_grid - sck_grid)
    rel_diff = abs_diff / (torch.abs(sck_grid) + 1e-12)  # Add small epsilon to avoid division by zero
    
    # Calculate statistics
    total_charge_cic = cic_grid.sum()
    total_charge_sck = sck_grid.sum()
    max_abs_diff = abs_diff.max()
    mean_abs_diff = abs_diff.mean()
    max_rel_diff = rel_diff.max()
    mean_rel_diff = rel_diff.mean()
    
    # Correlation coefficient
    correlation = torch.corrcoef(torch.stack([cic_grid.flatten(), sck_grid.flatten()]))[0, 1]
    
    results = {
        'total_charge_cic': total_charge_cic.item(),
        'total_charge_sck': total_charge_sck.item(),
        'charge_conservation_error': abs(total_charge_cic - total_charge_sck).item(),
        'max_absolute_difference': max_abs_diff.item(),
        'mean_absolute_difference': mean_abs_diff.item(),
        'max_relative_difference': max_rel_diff.item(),
        'mean_relative_difference': mean_rel_diff.item(),
        'correlation_coefficient': correlation.item(),
        'cic_grid_shape': cic_grid.shape,
        'sck_grid_shape': sck_grid.shape
    }
    
    print(f"Total charge (CIC): {results['total_charge_cic']:.6e} C")
    print(f"Total charge (SCK): {results['total_charge_sck']:.6e} C")
    print(f"Charge conservation error: {results['charge_conservation_error']:.6e} C")
    print(f"Max absolute difference: {results['max_absolute_difference']:.6e}")
    print(f"Mean absolute difference: {results['mean_absolute_difference']:.6e}")
    print(f"Max relative difference: {results['max_relative_difference']:.6e}")
    print(f"Mean relative difference: {results['mean_relative_difference']:.6e}")
    print(f"Correlation coefficient: {results['correlation_coefficient']:.6f}")
    
    return results, cic_grid, sck_grid, beam, grid_dimensions

# Run accuracy comparison
accuracy_results, cic_output, sck_output, beam, grid_dimensions = compare_charge_deposition_accuracy(
    grid_shape=(16, 16, 16), num_particles=1, offset=torch.tensor(2.5e-3)
)

# Check if results are practically identical
are_identical = torch.allclose(cic_output, sck_output, rtol=1e-5, atol=1e-12)
print(f"\nOutputs are practically identical (rtol=1e-5, atol=1e-12): {are_identical}")

# Run additional accuracy comparison with more particles for better visualization
print("\n" + "="*60)
print("ADDITIONAL COMPARISON WITH MORE PARTICLES")
print("="*60)
accuracy_results_multi, cic_output_multi, sck_output_multi, beam_multi, grid_dimensions_multi = compare_charge_deposition_accuracy(
    grid_shape=(16, 16, 16), num_particles=100, offset=torch.tensor(2.5e-3)
)

## 6. Visual Comparison of Charge Grids

In [ ]:
def plot_charge_grid_projections(cic_grid, sck_grid, grid_dimensions=None, particle_positions=None):
    """Create visualizations comparing charge grids using 1D and 2D projections along xyz axes"""
    
    # Convert to numpy and remove batch dimension
    cic_np = cic_grid[0].cpu().numpy()  
    sck_np = sck_grid[0].cpu().numpy()
    
    # Extract particle positions if provided
    if particle_positions is not None:
        particles_np = particle_positions[0].cpu().numpy()  # Remove batch dimension
        particle_x = particles_np[:, 0]  # x positions
        particle_y = particles_np[:, 2]  # y positions  
        particle_z = particles_np[:, 4]  # z positions (tau)
    
    # Calculate projections by summing along each axis
    # 2D projections
    cic_proj_xy = np.sum(cic_np, axis=2)  # Sum along Z -> X-Y projection
    sck_proj_xy = np.sum(sck_np, axis=2)
    
    cic_proj_xz = np.sum(cic_np, axis=1)  # Sum along Y -> X-Z projection
    sck_proj_xz = np.sum(sck_np, axis=1)
    
    cic_proj_yz = np.sum(cic_np, axis=0)  # Sum along X -> Y-Z projection
    sck_proj_yz = np.sum(sck_np, axis=0)
    
    # 1D projections
    cic_proj_x = np.sum(cic_np, axis=(1, 2))  # Sum along Y,Z -> X projection
    sck_proj_x = np.sum(sck_np, axis=(1, 2))
    
    cic_proj_y = np.sum(cic_np, axis=(0, 2))  # Sum along X,Z -> Y projection
    sck_proj_y = np.sum(sck_np, axis=(0, 2))
    
    cic_proj_z = np.sum(cic_np, axis=(0, 1))  # Sum along X,Y -> Z projection
    sck_proj_z = np.sum(sck_np, axis=(0, 1))
    
    # Create coordinate arrays
    if grid_dimensions is not None:
        # Use actual physical coordinates
        grid_dims = grid_dimensions[0].cpu().numpy()
        x_coords = np.linspace(-grid_dims[0], grid_dims[0], cic_np.shape[0] + 1)
        y_coords = np.linspace(-grid_dims[1], grid_dims[1], cic_np.shape[1] + 1)
        z_coords = np.linspace(-grid_dims[2], grid_dims[2], cic_np.shape[2] + 1)
        
        # Centers for line plots
        x_centers = (x_coords[:-1] + x_coords[1:]) / 2
        y_centers = (y_coords[:-1] + y_coords[1:]) / 2
        z_centers = (z_coords[:-1] + z_coords[1:]) / 2
        
        # Meshgrids for 2D plots
        X_xy, Y_xy = np.meshgrid(x_coords, y_coords)
        X_xz, Z_xz = np.meshgrid(x_coords, z_coords)
        Y_yz, Z_yz = np.meshgrid(y_coords, z_coords)
        
        # Labels with units
        x_label = 'X Position (m)'
        y_label = 'Y Position (m)'
        z_label = 'Z Position (m)'
        density_label = 'Charge Density (C/m³)'
        integrated_label = 'Integrated Charge (C)'
    else:
        # Use grid indices
        x_coords = np.arange(cic_np.shape[0] + 1)
        y_coords = np.arange(cic_np.shape[1] + 1)
        z_coords = np.arange(cic_np.shape[2] + 1)
        
        x_centers = np.arange(cic_np.shape[0])
        y_centers = np.arange(cic_np.shape[1])
        z_centers = np.arange(cic_np.shape[2])
        
        X_xy, Y_xy = np.meshgrid(x_coords, y_coords)
        X_xz, Z_xz = np.meshgrid(x_coords, z_coords)
        Y_yz, Z_yz = np.meshgrid(y_coords, z_coords)
        
        x_label = 'X Grid Index'
        y_label = 'Y Grid Index'
        z_label = 'Z Grid Index'
        density_label = 'Charge Density'
        integrated_label = 'Integrated Charge'
    
    # Create comprehensive figure with 2D projections and 1D profiles
    fig = plt.figure(figsize=(20, 16))
    
    # ===============================
    # 2D PROJECTIONS (3x3 grid)
    # ===============================
    
    # X-Y projections (top row)
    ax1 = plt.subplot(4, 3, 1)
    im1 = ax1.pcolormesh(X_xy, Y_xy, cic_proj_xy, cmap='plasma', shading='flat')
    ax1.set_title('CIC: X-Y Projection (summed over Z)')
    ax1.set_xlabel(x_label)
    ax1.set_ylabel(y_label)
    ax1.set_aspect('equal')
    plt.colorbar(im1, ax=ax1, label=integrated_label)
    
    # Overlay particles in X-Y projection
    if particle_positions is not None:
        ax1.scatter(particle_x, particle_y, c='red', s=8, alpha=0.6, 
                   edgecolors='white', linewidth=0.3, label=f'Particles (n={len(particle_x)})')
        ax1.legend()
    
    ax2 = plt.subplot(4, 3, 2)
    im2 = ax2.pcolormesh(X_xy, Y_xy, sck_proj_xy, cmap='plasma', shading='flat')
    ax2.set_title('SpaceChargeKick: X-Y Projection (summed over Z)')
    ax2.set_xlabel(x_label)
    ax2.set_ylabel(y_label)
    ax2.set_aspect('equal')
    plt.colorbar(im2, ax=ax2, label=integrated_label)
    
    # Overlay particles
    if particle_positions is not None:
        ax2.scatter(particle_x, particle_y, c='red', s=8, alpha=0.6, 
                   edgecolors='white', linewidth=0.3, label=f'Particles (n={len(particle_x)})')
        ax2.legend()
    
    ax3 = plt.subplot(4, 3, 3)
    diff_xy = np.abs(cic_proj_xy - sck_proj_xy)
    im3 = ax3.pcolormesh(X_xy, Y_xy, diff_xy, cmap='Reds', shading='flat')
    ax3.set_title('Absolute Difference: X-Y Projection')
    ax3.set_xlabel(x_label)
    ax3.set_ylabel(y_label)
    ax3.set_aspect('equal')
    plt.colorbar(im3, ax=ax3, label=integrated_label)
    
    # X-Z projections (second row)
    ax4 = plt.subplot(4, 3, 4)
    im4 = ax4.pcolormesh(X_xz, Z_xz, cic_proj_xz, cmap='plasma', shading='flat')
    ax4.set_title('CIC: X-Z Projection (summed over Y)')
    ax4.set_xlabel(x_label)
    ax4.set_ylabel(z_label)
    ax4.set_aspect('equal')
    plt.colorbar(im4, ax=ax4, label=integrated_label)
    
    # Overlay particles
    if particle_positions is not None:
        ax4.scatter(particle_x, particle_z, c='red', s=8, alpha=0.6, 
                   edgecolors='white', linewidth=0.3, label=f'Particles (n={len(particle_x)})')
        ax4.legend()
    
    ax5 = plt.subplot(4, 3, 5)
    im5 = ax5.pcolormesh(X_xz, Z_xz, sck_proj_xz, cmap='plasma', shading='flat')
    ax5.set_title('SpaceChargeKick: X-Z Projection (summed over Y)')
    ax5.set_xlabel(x_label)
    ax5.set_ylabel(z_label)
    ax5.set_aspect('equal')
    plt.colorbar(im5, ax=ax5, label=integrated_label)
    
    # Overlay particles
    if particle_positions is not None:
        ax5.scatter(particle_x, particle_z, c='red', s=8, alpha=0.6, 
                   edgecolors='white', linewidth=0.3, label=f'Particles (n={len(particle_x)})')
        ax5.legend()
    
    ax6 = plt.subplot(4, 3, 6)
    diff_xz = np.abs(cic_proj_xz - sck_proj_xz)
    im6 = ax6.pcolormesh(X_xz, Z_xz, diff_xz, cmap='Reds', shading='flat')
    ax6.set_title('Absolute Difference: X-Z Projection')
    ax6.set_xlabel(x_label)
    ax6.set_ylabel(z_label)
    ax6.set_aspect('equal')
    plt.colorbar(im6, ax=ax6, label=integrated_label)
    
    # Y-Z projections (third row)
    ax7 = plt.subplot(4, 3, 7)
    im7 = ax7.pcolormesh(Y_yz, Z_yz, cic_proj_yz, cmap='plasma', shading='flat')
    ax7.set_title('CIC: Y-Z Projection (summed over X)')
    ax7.set_xlabel(y_label)
    ax7.set_ylabel(z_label)
    ax7.set_aspect('equal')
    plt.colorbar(im7, ax=ax7, label=integrated_label)
    
    # Overlay particles
    if particle_positions is not None:
        ax7.scatter(particle_y, particle_z, c='red', s=8, alpha=0.6, 
                   edgecolors='white', linewidth=0.3, label=f'Particles (n={len(particle_x)})')
        ax7.legend()
    
    ax8 = plt.subplot(4, 3, 8)
    im8 = ax8.pcolormesh(Y_yz, Z_yz, sck_proj_yz, cmap='plasma', shading='flat')
    ax8.set_title('SpaceChargeKick: Y-Z Projection (summed over X)')
    ax8.set_xlabel(y_label)
    ax8.set_ylabel(z_label)
    ax8.set_aspect('equal')
    plt.colorbar(im8, ax=ax8, label=integrated_label)
    
    # Overlay particles
    if particle_positions is not None:
        ax8.scatter(particle_y, particle_z, c='red', s=8, alpha=0.6, 
                   edgecolors='white', linewidth=0.3, label=f'Particles (n={len(particle_x)})')
        ax8.legend()
    
    ax9 = plt.subplot(4, 3, 9)
    diff_yz = np.abs(cic_proj_yz - sck_proj_yz)
    im9 = ax9.pcolormesh(Y_yz, Z_yz, diff_yz, cmap='Reds', shading='flat')
    ax9.set_title('Absolute Difference: Y-Z Projection')
    ax9.set_xlabel(y_label)
    ax9.set_ylabel(z_label)
    ax9.set_aspect('equal')
    plt.colorbar(im9, ax=ax9, label=integrated_label)
    
    # ===============================
    # 1D PROJECTIONS (bottom row)
    # ===============================
    
    # X-direction projection
    ax10 = plt.subplot(4, 3, 10)
    ax10.plot(x_centers, cic_proj_x, 'b-', linewidth=3, label='CIC', alpha=0.8)
    ax10.plot(x_centers, sck_proj_x, 'r--', linewidth=3, label='SpaceChargeKick', alpha=0.8)
    ax10.fill_between(x_centers, 0, cic_proj_x, alpha=0.3, color='blue')
    ax10.fill_between(x_centers, 0, sck_proj_x, alpha=0.3, color='red')
    ax10.set_xlabel(x_label)
    ax10.set_ylabel(integrated_label)
    ax10.set_title('X-direction Projection (summed over Y,Z)')
    ax10.legend()
    ax10.grid(True, alpha=0.3)
    
    # Y-direction projection
    ax11 = plt.subplot(4, 3, 11)
    ax11.plot(y_centers, cic_proj_y, 'b-', linewidth=3, label='CIC', alpha=0.8)
    ax11.plot(y_centers, sck_proj_y, 'r--', linewidth=3, label='SpaceChargeKick', alpha=0.8)
    ax11.fill_between(y_centers, 0, cic_proj_y, alpha=0.3, color='blue')
    ax11.fill_between(y_centers, 0, sck_proj_y, alpha=0.3, color='red')
    ax11.set_xlabel(y_label)
    ax11.set_ylabel(integrated_label)
    ax11.set_title('Y-direction Projection (summed over X,Z)')
    ax11.legend()
    ax11.grid(True, alpha=0.3)
    
    # Z-direction projection
    ax12 = plt.subplot(4, 3, 12)
    ax12.plot(z_centers, cic_proj_z, 'b-', linewidth=3, label='CIC', alpha=0.8)
    ax12.plot(z_centers, sck_proj_z, 'r--', linewidth=3, label='SpaceChargeKick', alpha=0.8)
    ax12.fill_between(z_centers, 0, cic_proj_z, alpha=0.3, color='blue')
    ax12.fill_between(z_centers, 0, sck_proj_z, alpha=0.3, color='red')
    ax12.set_xlabel(z_label)
    ax12.set_ylabel(integrated_label)
    ax12.set_title('Z-direction Projection (summed over X,Y)')
    ax12.legend()
    ax12.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # ===============================
    # QUANTITATIVE COMPARISON
    # ===============================
    
    print("\nQUANTITATIVE PROJECTION COMPARISON:")
    print("=" * 60)
    
    # Calculate correlation coefficients for each projection
    corr_xy = np.corrcoef(cic_proj_xy.flatten(), sck_proj_xy.flatten())[0, 1]
    corr_xz = np.corrcoef(cic_proj_xz.flatten(), sck_proj_xz.flatten())[0, 1]
    corr_yz = np.corrcoef(cic_proj_yz.flatten(), sck_proj_yz.flatten())[0, 1]
    
    corr_x = np.corrcoef(cic_proj_x, sck_proj_x)[0, 1]
    corr_y = np.corrcoef(cic_proj_y, sck_proj_y)[0, 1]
    corr_z = np.corrcoef(cic_proj_z, sck_proj_z)[0, 1]
    
    # Calculate relative differences
    rel_diff_xy = np.abs(diff_xy) / (np.abs(cic_proj_xy) + np.abs(sck_proj_xy) + 1e-12)
    rel_diff_xz = np.abs(diff_xz) / (np.abs(cic_proj_xz) + np.abs(sck_proj_xz) + 1e-12)
    rel_diff_yz = np.abs(diff_yz) / (np.abs(cic_proj_yz) + np.abs(sck_proj_yz) + 1e-12)
    
    diff_x = np.abs(cic_proj_x - sck_proj_x)
    diff_y = np.abs(cic_proj_y - sck_proj_y)
    diff_z = np.abs(cic_proj_z - sck_proj_z)
    
    rel_diff_x = diff_x / (np.abs(cic_proj_x) + np.abs(sck_proj_x) + 1e-12)
    rel_diff_y = diff_y / (np.abs(cic_proj_y) + np.abs(sck_proj_y) + 1e-12)
    rel_diff_z = diff_z / (np.abs(cic_proj_z) + np.abs(sck_proj_z) + 1e-12)
    
    print("\n2D PROJECTION CORRELATIONS:")
    print(f"  X-Y plane: {corr_xy:.8f}")
    print(f"  X-Z plane: {corr_xz:.8f}")
    print(f"  Y-Z plane: {corr_yz:.8f}")
    
    print("\n1D PROJECTION CORRELATIONS:")
    print(f"  X-direction: {corr_x:.8f}")
    print(f"  Y-direction: {corr_y:.8f}")
    print(f"  Z-direction: {corr_z:.8f}")
    
    print("\n2D PROJECTION MAX RELATIVE DIFFERENCES:")
    print(f"  X-Y plane: {rel_diff_xy.max():.6e}")
    print(f"  X-Z plane: {rel_diff_xz.max():.6e}")
    print(f"  Y-Z plane: {rel_diff_yz.max():.6e}")
    
    print("\n1D PROJECTION MAX RELATIVE DIFFERENCES:")
    print(f"  X-direction: {rel_diff_x.max():.6e}")
    print(f"  Y-direction: {rel_diff_y.max():.6e}")
    print(f"  Z-direction: {rel_diff_z.max():.6e}")
    
    print("\nTOTAL CHARGE COMPARISON:")
    total_cic = np.sum(cic_np)
    total_sck = np.sum(sck_np)
    print(f"  CIC total: {total_cic:.6e}")
    print(f"  SCK total: {total_sck:.6e}")
    print(f"  Relative difference: {abs(total_cic - total_sck) / (total_cic + total_sck + 1e-12):.6e}")

# Create projection-based visualizations
print("\nPROJECTION-BASED CHARGE GRID COMPARISON:")
print("=" * 60)
print("Single particle case:")
plot_charge_grid_projections(cic_output, sck_output, grid_dimensions, beam.particles)

print("\nMultiple particle case:")
grid_dimensions_multi, _ = TestDataGenerator.create_grid_parameters((16, 16, 16), beam_multi)
plot_charge_grid_projections(cic_output_multi, sck_output_multi, grid_dimensions_multi, beam_multi.particles)

In [ ]:
grid_dimensions

## 7. Performance Scaling Study

In [ ]:
def run_comprehensive_scaling_study():
    """
    Comprehensive scaling study with statistical analysis and confidence intervals
    """
    
    # Test different numbers of particles (reduced for faster testing)
    particle_counts = [1000, 5000, 10000, 25000, 50000, 100000, 250000, 500000, 1000000]
    grid_sizes = [(16, 16, 16), (32, 32, 32), (64, 64, 64), (128, 128, 128)]
    
    results_df = []
    
    print("Running comprehensive scaling study with statistical analysis...")
    print("=" * 70)
    
    total_combinations = len(grid_sizes) * len(particle_counts)
    current_combination = 0
    
    for grid_shape in grid_sizes:
        print(f"\nTesting grid shape: {grid_shape}")
        print(f"{'-'*50}")
        
        for num_particles in particle_counts:
            current_combination += 1
            progress = (current_combination / total_combinations) * 100
            
            print(f"  [{progress:5.1f}%] {num_particles:,} particles...", end=" ", flush=True)
            
            try:
                # Test CIC method (3D only for scaling study)
                cic_results = benchmark_cic_methods(
                    num_particles, grid_shape, batch_size=1, num_trials=5
                )
                cic_3d_result = cic_results[0]  # Take the 3D result
                
                # Test SpaceChargeKick method
                sck_result = benchmark_space_charge_kick_method(
                    num_particles, grid_shape, batch_size=1, num_trials=5
                )
                
                # Calculate speedup with uncertainty propagation
                speedup_mean = sck_result.execution_time_mean / cic_3d_result.execution_time_mean
                speedup_rel_error = np.sqrt(
                    (sck_result.execution_time_std / sck_result.execution_time_mean)**2 + 
                    (cic_3d_result.execution_time_std / cic_3d_result.execution_time_mean)**2
                )
                speedup_std = speedup_mean * speedup_rel_error
                
                # Memory ratio calculations
                memory_ratio_mean = cic_3d_result.memory_usage_mean / sck_result.memory_usage_mean if sck_result.memory_usage_mean > 0 else np.inf
                
                # Store comprehensive results
                results_df.append({
                    'num_particles': num_particles,
                    'grid_shape': str(grid_shape),
                    'grid_volume': np.prod(grid_shape),
                    
                    # CIC timing statistics
                    'cic_time_mean': cic_3d_result.execution_time_mean,
                    'cic_time_std': cic_3d_result.execution_time_std,
                    'cic_time_min': cic_3d_result.execution_time_min,
                    'cic_time_max': cic_3d_result.execution_time_max,
                    
                    # CIC memory statistics
                    'cic_memory_mean': cic_3d_result.memory_usage_mean,
                    'cic_memory_std': cic_3d_result.memory_usage_std,
                    
                    # SpaceChargeKick timing statistics
                    'sck_time_mean': sck_result.execution_time_mean,
                    'sck_time_std': sck_result.execution_time_std,
                    'sck_time_min': sck_result.execution_time_min,
                    'sck_time_max': sck_result.execution_time_max,
                    
                    # SpaceChargeKick memory statistics
                    'sck_memory_mean': sck_result.memory_usage_mean,
                    'sck_memory_std': sck_result.memory_usage_std,
                    
                    # Performance metrics
                    'speedup_mean': speedup_mean,
                    'speedup_std': speedup_std,
                    'memory_ratio_mean': memory_ratio_mean,
                    
                    # Coefficient of variation (relative std dev)
                    'cic_time_cv': cic_3d_result.execution_time_std / cic_3d_result.execution_time_mean * 100,
                    'sck_time_cv': sck_result.execution_time_std / sck_result.execution_time_mean * 100,
                    
                    'num_trials': cic_3d_result.num_trials
                })
                
                print(f"CIC: {cic_3d_result.execution_time_mean:.3f}±{cic_3d_result.execution_time_std:.3f}s, "
                      f"SCK: {sck_result.execution_time_mean:.3f}±{sck_result.execution_time_std:.3f}s, "
                      f"Speedup: {speedup_mean:.1f}±{speedup_std:.1f}x")
                
            except Exception as e:
                print(f"Error: {e}")
                continue
    
    return results_df

def analyze_scaling_results(df):
    """Analyze and summarize scaling study results"""
    
    if len(df) == 0:
        print("No results to analyze")
        return
    
    print(f"\n{'='*80}")
    print("SCALING STUDY ANALYSIS")
    print(f"{'='*80}")
    
    # Summary statistics
    print(f"\nOVERALL PERFORMANCE STATISTICS:")
    print(f"{'-'*40}")
    print(f"Mean speedup: {df['speedup_mean'].mean():.2f} ± {df['speedup_std'].mean():.2f}")
    print(f"Max speedup: {df['speedup_mean'].max():.2f}")
    print(f"Min speedup: {df['speedup_mean'].min():.2f}")
    print(f"Speedup std dev: {df['speedup_mean'].std():.2f}")
    
    # Timing variability analysis
    print(f"\nTIMING VARIABILITY (Coefficient of Variation):")
    print(f"{'-'*45}")
    print(f"CIC timing CV: {df['cic_time_cv'].mean():.1f}% ± {df['cic_time_cv'].std():.1f}%")
    print(f"SCK timing CV: {df['sck_time_cv'].mean():.1f}% ± {df['sck_time_cv'].std():.1f}%")
    
    # Performance by grid size
    print(f"\nPERFORMANCE BY GRID SIZE:")
    print(f"{'-'*30}")
    for grid_shape in df['grid_shape'].unique():
        grid_data = df[df['grid_shape'] == grid_shape]
        mean_speedup = grid_data['speedup_mean'].mean()
        std_speedup = grid_data['speedup_mean'].std()
        print(f"Grid {grid_shape}: {mean_speedup:.2f} ± {std_speedup:.2f}x speedup")
    
    # Memory efficiency analysis
    print(f"\nMEMORY EFFICIENCY:")
    print(f"{'-'*20}")
    print(f"Mean memory ratio (CIC/SCK): {df['memory_ratio_mean'].mean():.2f}")
    print(f"CIC memory usage range: {df['cic_memory_mean'].min():.1f} - {df['cic_memory_mean'].max():.1f} MB")
    print(f"SCK memory usage range: {df['sck_memory_mean'].min():.1f} - {df['sck_memory_mean'].max():.1f} MB")

# Run the comprehensive scaling study
scaling_results = run_comprehensive_scaling_study()

# Convert to DataFrame for analysis
import pandas as pd
df = pd.DataFrame(scaling_results)

# Analyze results
analyze_scaling_results(df)

# Display detailed results table
if len(scaling_results) > 0:
    print(f"\n{'DETAILED RESULTS TABLE:'}")
    print(f"{'-'*25}")
    display_columns = ['num_particles', 'grid_shape', 'cic_time_mean', 'cic_time_std', 
                      'sck_time_mean', 'sck_time_std', 'speedup_mean', 'speedup_std']
    print(df[display_columns].to_string(index=False, float_format='%.4f'))

## 8. Performance Scaling Visualizations

In [ ]:
def plot_scaling_results_with_error_bars(df):
    """Create visualizations with error bars and confidence intervals"""
    
    if len(df) == 0:
        print("No data to plot")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Execution time vs number of particles (with error bars)
    for grid_shape in df['grid_shape'].unique():
        grid_data = df[df['grid_shape'] == grid_shape]
        axes[0,0].errorbar(grid_data['num_particles'], grid_data['cic_time_mean'], 
                          yerr=grid_data['cic_time_std'], fmt='o-', 
                          label=f'CIC {grid_shape}', linewidth=2, capsize=5)
        axes[0,0].errorbar(grid_data['num_particles'], grid_data['sck_time_mean'], 
                          yerr=grid_data['sck_time_std'], fmt='s--', 
                          label=f'SCK {grid_shape}', linewidth=2, capsize=5)
    
    axes[0,0].set_xscale('log')
    axes[0,0].set_yscale('log')
    axes[0,0].set_xlabel('Number of Particles')
    axes[0,0].set_ylabel('Execution Time (s)')
    axes[0,0].set_title('Execution Time Scaling (with error bars)')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # Plot 2: Speedup factor with confidence intervals
    for grid_shape in df['grid_shape'].unique():
        grid_data = df[df['grid_shape'] == grid_shape]
        axes[0,1].errorbar(grid_data['num_particles'], grid_data['speedup_mean'], 
                          yerr=grid_data['speedup_std'], fmt='o-', 
                          label=f'{grid_shape}', linewidth=2, capsize=5)
    
    axes[0,1].axhline(y=1, color='red', linestyle=':', alpha=0.7, label='Equal Performance')
    axes[0,1].set_xscale('log')
    axes[0,1].set_xlabel('Number of Particles')
    axes[0,1].set_ylabel('Speedup Factor (SCK Time / CIC Time)')
    axes[0,1].set_title('CIC Performance Advantage (with confidence intervals)')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Plot 3: Memory usage comparison with error bars
    for grid_shape in df['grid_shape'].unique():
        grid_data = df[df['grid_shape'] == grid_shape]
        axes[0,2].errorbar(grid_data['num_particles'], grid_data['cic_memory_mean'], 
                          yerr=grid_data['cic_memory_std'], fmt='o-', 
                          label=f'CIC {grid_shape}', linewidth=2, capsize=5)
        axes[0,2].errorbar(grid_data['num_particles'], grid_data['sck_memory_mean'], 
                          yerr=grid_data['sck_memory_std'], fmt='s--', 
                          label=f'SCK {grid_shape}', linewidth=2, capsize=5)
    
    axes[0,2].set_xscale('log')
    axes[0,2].set_yscale('log')
    axes[0,2].set_xlabel('Number of Particles')
    axes[0,2].set_ylabel('Memory Usage (MB)')
    axes[0,2].set_title('Memory Usage Scaling (with error bars)')
    axes[0,2].legend()
    axes[0,2].grid(True, alpha=0.3)
    
    # Plot 4: Coefficient of variation (timing stability)
    width = 0.35
    grid_shapes = df['grid_shape'].unique()
    x_pos = np.arange(len(grid_shapes))
    
    cic_cv_means = [df[df['grid_shape'] == gs]['cic_time_cv'].mean() for gs in grid_shapes]
    sck_cv_means = [df[df['grid_shape'] == gs]['sck_time_cv'].mean() for gs in grid_shapes]
    
    axes[1,0].bar(x_pos - width/2, cic_cv_means, width, label='CIC', alpha=0.8)
    axes[1,0].bar(x_pos + width/2, sck_cv_means, width, label='SCK', alpha=0.8)
    
    axes[1,0].set_xlabel('Grid Shape')
    axes[1,0].set_ylabel('Coefficient of Variation (%)')
    axes[1,0].set_title('Timing Stability (lower is more stable)')
    axes[1,0].set_xticks(x_pos)
    axes[1,0].set_xticklabels(grid_shapes)
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # Plot 5: Performance scaling efficiency
    if len(df['grid_volume'].unique()) > 1:
        grid_volumes = df['grid_volume'].unique()
        mean_speedup = []
        std_speedup = []
        
        for vol in sorted(grid_volumes):
            vol_data = df[df['grid_volume'] == vol]
            mean_speedup.append(vol_data['speedup_mean'].mean())
            std_speedup.append(vol_data['speedup_mean'].std())
        
        axes[1,1].errorbar(sorted(grid_volumes), mean_speedup, yerr=std_speedup,
                          fmt='bo-', linewidth=2, markersize=8, capsize=5)
        axes[1,1].set_xscale('log')
        axes[1,1].set_xlabel('Grid Volume (Total Grid Points)')
        axes[1,1].set_ylabel('Mean Speedup Factor')
        axes[1,1].set_title('Performance vs Grid Size')
        axes[1,1].grid(True, alpha=0.3)
    else:
        axes[1,1].text(0.5, 0.5, 'Need multiple grid sizes\nfor this analysis', 
                      ha='center', va='center', transform=axes[1,1].transAxes)
        axes[1,1].set_title('Performance vs Grid Size (insufficient data)')
    
    # Plot 6: Execution time distribution (violin plot or box plot)
    if len(df) > 0:
        # Create box plot showing distribution of execution times
        cic_times_by_grid = [df[df['grid_shape'] == gs]['cic_time_mean'].values for gs in grid_shapes]
        sck_times_by_grid = [df[df['grid_shape'] == gs]['sck_time_mean'].values for gs in grid_shapes]
        
        # Combine data for box plot
        all_times = []
        labels = []
        for i, gs in enumerate(grid_shapes):
            all_times.extend([cic_times_by_grid[i], sck_times_by_grid[i]])
            labels.extend([f'CIC\n{gs}', f'SCK\n{gs}'])
        
        bp = axes[1,2].boxplot(all_times, labels=labels, patch_artist=True)
        
        # Color the boxes
        colors = ['lightblue', 'lightcoral'] * len(grid_shapes)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        
        axes[1,2].set_ylabel('Execution Time (s)')
        axes[1,2].set_title('Execution Time Distribution')
        axes[1,2].tick_params(axis='x', rotation=45)
        axes[1,2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics display
    print("\nSUMMARY STATISTICS:")
    print("=" * 50)
    print(f"Overall mean speedup: {df['speedup_mean'].mean():.2f} ± {df['speedup_std'].mean():.2f}")
    print(f"Best speedup achieved: {df['speedup_mean'].max():.2f}")
    print(f"Most consistent method (lowest CV): {'CIC' if df['cic_time_cv'].mean() < df['sck_time_cv'].mean() else 'SCK'}")
    print(f"Memory efficiency: CIC uses {df['memory_ratio_mean'].mean():.2f}x memory vs SCK")

# Create enhanced visualizations with error bars
if len(scaling_results) > 0:
    plot_scaling_results_with_error_bars(df)
else:
    print("No scaling results to plot - run the scaling study first")